In [1]:
!pip install transformers torch -q

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
drive_path = '/content/drive/MyDrive/FYP_DATA/'

In [5]:
finbert = pipeline(
    task             = 'text-classification',
    model            = 'ProsusAI/finbert',
    return_all_scores = True,
    device           = -1
)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
df_tweets = pd.read_excel(
    drive_path + 'Bitcoin NLP Data.xlsx'
)

In [7]:
df_tweets.head(5)

,tweet_id,text,user_info.screen_name,user_info.name,user_info.verified,created_at,retweets,favorites,replies,quotes,views,lang,url
0,947722684216918016,Happy New Years from @ShapeShift_io #bitcoin #...,ErikVoorhees,Erik Voorhees,True,2018-01-01 06:54:30,23,224,20,4,NaN,en,https://twitter.com/ErikVoorhees/status/947722...
1,947860681239522048,"If you haven't already, I would go peep @lopp'...",MartyBent,Marty Bent,True,2018-01-01 16:02:51,4,25,2,0,NaN,en,https://twitter.com/MartyBent/status/947860681...
2,948205583810866048,CoinDesk: Erik Voorhees on ShapeShift and Dig...,ErikVoorhees,Erik Voorhees,True,2018-01-02 14:53:22,8,26,0,4,NaN,nl,https://twitter.com/ErikVoorhees/status/948205...
3,948218300709981056,Green day in the crypto markets to kick off 20...,ErikVoorhees,Erik Voorhees,True,2018-01-02 15:43:54,25,123,4,2,NaN,en,https://twitter.com/ErikVoorhees/status/948218...
4,948319938502119040,Just passed 100k subscribers on my YouTube cha...,aantonop,Andreas (aantonop Team),True,2018-01-02 22:27:46,161,1832,102,11,NaN,en,https://twitter.com/aantonop/status/9483199385...


In [8]:
df_tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10415 entries, 0 to 10414
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   tweet_id               10415 non-null  int64         
 1   text                   10415 non-null  object        
 2   user_info.screen_name  10415 non-null  object        
 3   user_info.name         10415 non-null  object        
 4   user_info.verified     10415 non-null  bool          
 5   created_at             10415 non-null  datetime64[ns]
 6   retweets               10415 non-null  int64         
 7   favorites              10415 non-null  int64         
 8   replies                10415 non-null  int64         
 9   quotes                 10415 non-null  int64         
 10  views                  2458 non-null   float64       
 11  lang                   10415 non-null  object        
 12  url                    10415 non-null  object        
dtypes

In [9]:
print(df_tweets['lang'].value_counts())

lang
en     10217
qme       52
und       35
es        25
qht       13
pt        12
de         9
nl         8
tl         6
ca         6
it         6
fr         6
da         5
et         4
cy         2
vi         1
lt         1
in         1
ht         1
hu         1
hi         1
no         1
sl         1
cs         1
Name: count, dtype: int64


In [10]:
print(df_tweets['user_info.screen_name'].value_counts())

user_info.screen_name
DocumentingBTC    3498
saylor            2704
100trillionUSD     938
aantonop           737
PrestonPysh        581
tyler              528
ErikVoorhees       428
cameron            341
willywoo           307
MartyBent          209
TuurDemeester       78
jack                57
gladstein            7
nic_carter           2
Name: count, dtype: int64


In [11]:
df_tweets['Date'] = pd.to_datetime(df_tweets['created_at']).dt.normalize()

In [12]:
df_tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10415 entries, 0 to 10414
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   tweet_id               10415 non-null  int64         
 1   text                   10415 non-null  object        
 2   user_info.screen_name  10415 non-null  object        
 3   user_info.name         10415 non-null  object        
 4   user_info.verified     10415 non-null  bool          
 5   created_at             10415 non-null  datetime64[ns]
 6   retweets               10415 non-null  int64         
 7   favorites              10415 non-null  int64         
 8   replies                10415 non-null  int64         
 9   quotes                 10415 non-null  int64         
 10  views                  2458 non-null   float64       
 11  lang                   10415 non-null  object        
 12  url                    10415 non-null  object        
 13  D

In [13]:
df_tweets['Date'].min().date()

datetime.date(2018, 1, 1)

In [14]:
df_tweets['Date'].max().date()

datetime.date(2026, 2, 10)

In [15]:
df_tweets['Date'].nunique()

2190

In [16]:
df_tweets.isnull().sum()

,0
tweet_id,0
text,0
user_info.screen_name,0
user_info.name,0
user_info.verified,0
created_at,0
retweets,0
favorites,0
replies,0
quotes,0


In [17]:
df_tweets[['retweets','favorites','replies']].describe().round(1)

,retweets,favorites,replies
count,10415.0,10415.0,10415.0
mean,522.3,3440.8,215.8
std,1024.4,7310.0,458.7
min,0.0,1.0,0.0
25%,79.0,511.0,30.0
50%,244.0,1554.0,71.0
75%,688.0,4504.0,238.0
max,68310.0,561275.0,30492.0


In [18]:
df_tweets['Date'] = pd.to_datetime(df_tweets['created_at']).dt.normalize()
df_en_check = df_tweets[df_tweets['lang'] == 'en'].copy()

print(f"{'Year':<6} {'Days w/tweets':>13} {'Total days':>10} {'Missing':>7} "
      f"{'Accounts':>8} {'Tweets':>6} {'Quality'}")
print("-" * 70)

for yr in range(2018, 2026):
    yd  = df_en_check[df_en_check['Date'].dt.year == yr]
    td  = 366 if yr in [2020, 2024] else 365
    cv  = yd['Date'].nunique()
    ac  = yd['user_info.screen_name'].nunique()
    tw  = len(yd)
    ms  = td - cv
    q   = ('EXCELLENT' if ms < 20  else
           'GOOD'      if ms < 60  else
           'FAIR'      if ms < 150 else 'POOR')
    print(f"{yr:<6} {cv:>13} {td:>10} {ms:>7} {ac:>8} {tw:>6}  {q}")

Year   Days w/tweets Total days Missing Accounts Tweets Quality
----------------------------------------------------------------------
2018             144        365     221        8    180  POOR
2019             218        365     147        7    333  FAIR
2020             326        366      40       13   1365  GOOD
2021             362        365       3       13   3833  EXCELLENT
2022             361        365       4       12   2174  EXCELLENT
2023             353        365      12       10   1373  EXCELLENT
2024             337        366      29       11    883  GOOD
2025              66        365     299        3     67  POOR


In [19]:
def clean_tweet(text):
    """
    Clean a single tweet for FinBERT sentiment analysis.

    What we remove and WHY each decision:
    """
    if pd.isna(text):
        return ''

    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#([A-Za-z0-9_]+)', r'\1', text)
    text = re.sub(r'\$([A-Za-z0-9]+)', r'\1', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^\w\s\.\,\!\?\-\%\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [20]:
test_tweets = [
    'Is #bitcoin in a bubble? Are #cryptocurrencies in a bubble? YES.',
    'MicroStrategy Adopts #Bitcoin as Primary Treasury Reserve Asset. https://t.co/abc',
    'Strategy has acquired 1,142 BTC for ~$90.0 million at ~$78,815 per bitcoin.',
    'Happy New Years from @ShapeShift_io #bitcoin #ethereum https://t.co/kNnLydLxvY',
    'If you sold your #Bitcoin then you probably should not have been in it',
    'Bitcoin getting ready to trend back below its 21 DMA. #bitcoin https://t.co/H9L',
]

for tweet in test_tweets:
    cleaned = clean_tweet(tweet)
    print(f"ORIGINAL: {tweet[:90]}")
    print(f"CLEANED:  {cleaned[:90]}")
    print(f"LENGTH:   {len(cleaned)} characters | Will keep: {len(cleaned) > 10}")
    print()

ORIGINAL: Is #bitcoin in a bubble? Are #cryptocurrencies in a bubble? YES.
CLEANED:  Is bitcoin in a bubble? Are cryptocurrencies in a bubble? YES.
LENGTH:   62 characters | Will keep: True

ORIGINAL: MicroStrategy Adopts #Bitcoin as Primary Treasury Reserve Asset. https://t.co/abc
CLEANED:  MicroStrategy Adopts Bitcoin as Primary Treasury Reserve Asset.
LENGTH:   63 characters | Will keep: True

ORIGINAL: Strategy has acquired 1,142 BTC for ~$90.0 million at ~$78,815 per bitcoin.
CLEANED:  Strategy has acquired 1,142 BTC for 90.0 million at 78,815 per bitcoin.
LENGTH:   71 characters | Will keep: True

ORIGINAL: Happy New Years from @ShapeShift_io #bitcoin #ethereum https://t.co/kNnLydLxvY
CLEANED:  Happy New Years from bitcoin ethereum
LENGTH:   37 characters | Will keep: True

ORIGINAL: If you sold your #Bitcoin then you probably should not have been in it
CLEANED:  If you sold your Bitcoin then you probably should not have been in it
LENGTH:   69 characters | Will keep: True

ORIGI

In [21]:
df_tweets['clean_text'] = df_tweets['text'].apply(clean_tweet)

In [22]:
df_tweets[['text', 'clean_text']].head()

,text,clean_text
0,Happy New Years from @ShapeShift_io #bitcoin #...,Happy New Years from bitcoin ethereum
1,"If you haven't already, I would go peep @lopp'...","If you haven't already, I would go peep 's twe..."
2,CoinDesk: Erik Voorhees on ShapeShift and Dig...,CoinDesk Erik Voorhees on ShapeShift and Digit...
3,Green day in the crypto markets to kick off 20...,Green day in the crypto markets to kick off 20...
4,Just passed 100k subscribers on my YouTube cha...,Just passed 100k subscribers on my YouTube cha...


In [23]:
df_en = df_tweets[df_tweets['lang'] == 'en'].copy()

In [24]:
df_en['lang'].value_counts()

,count
lang,
en,10217


In [25]:
df_en = df_en[df_en['clean_text'].str.len() > 10].copy()

In [26]:
df_en['clean_text'].str.len().describe()

,clean_text
count,10217.000000
mean,125.140452
std,89.037549
min,12.000000
25%,53.000000
50%,106.000000
75%,192.000000
max,2629.000000


In [27]:
df_en = df_en.reset_index(drop=True)

In [28]:
df_en.head()

,tweet_id,text,user_info.screen_name,user_info.name,user_info.verified,created_at,retweets,favorites,replies,quotes,views,lang,url,Date,clean_text
0,947722684216918016,Happy New Years from @ShapeShift_io #bitcoin #...,ErikVoorhees,Erik Voorhees,True,2018-01-01 06:54:30,23,224,20,4,NaN,en,https://twitter.com/ErikVoorhees/status/947722...,2018-01-01,Happy New Years from bitcoin ethereum
1,947860681239522048,"If you haven't already, I would go peep @lopp'...",MartyBent,Marty Bent,True,2018-01-01 16:02:51,4,25,2,0,NaN,en,https://twitter.com/MartyBent/status/947860681...,2018-01-01,"If you haven't already, I would go peep 's twe..."
2,948218300709981056,Green day in the crypto markets to kick off 20...,ErikVoorhees,Erik Voorhees,True,2018-01-02 15:43:54,25,123,4,2,NaN,en,https://twitter.com/ErikVoorhees/status/948218...,2018-01-02,Green day in the crypto markets to kick off 20...
3,948319938502119040,Just passed 100k subscribers on my YouTube cha...,aantonop,Andreas (aantonop Team),True,2018-01-02 22:27:46,161,1832,102,11,NaN,en,https://twitter.com/aantonop/status/9483199385...,2018-01-02,Just passed 100k subscribers on my YouTube cha...
4,949840833376694016,This guy cryptos.\n 88N8 Digital Gold (Prod. ...,ErikVoorhees,Erik Voorhees,True,2018-01-07 03:11:16,69,181,25,14,NaN,en,https://twitter.com/ErikVoorhees/status/949840...,2018-01-07,This guy cryptos. 88N8 Digital Gold Prod. Pyro...


In [29]:
df_en['Date'] = pd.to_datetime(df_en['created_at']).dt.normalize()

In [30]:
df_en[['created_at', 'Date']].head()

,created_at,Date
0,2018-01-01 06:54:30,2018-01-01
1,2018-01-01 16:02:51,2018-01-01
2,2018-01-02 15:43:54,2018-01-02
3,2018-01-02 22:27:46,2018-01-02
4,2018-01-07 03:11:16,2018-01-07


In [31]:
df_en['engagement_weight'] = (
    df_en['retweets'].fillna(0)  * 2.0 +
    df_en['favorites'].fillna(0) * 1.0 +
    df_en['replies'].fillna(0)   * 1.5
).clip(lower=1)

In [32]:
df_en[['retweets','favorites','replies','engagement_weight']].head()

,retweets,favorites,replies,engagement_weight
0,23,224,20,300.0
1,4,25,2,36.0
2,25,123,4,179.0
3,161,1832,102,2307.0
4,69,181,25,356.5


In [33]:
df_en['engagement_weight'].describe().round(1)

,engagement_weight
count,10217.0
mean,4793.0
std,9114.6
min,2.0
25%,752.5
50%,2188.0
75%,6216.5
max,585124.0


In [34]:
top5 = df_en.nlargest(5, 'engagement_weight')[
    ['Date','user_info.screen_name','clean_text','engagement_weight']
]
top5

,Date,user_info.screen_name,clean_text,engagement_weight
2749,2021-02-21,DocumentingBTC,Explaining bitcoin at 100 to an empty room.,585124.0
1202,2020-10-14,jack,Donate via Bitcoin to help EndSARS,214455.5
1120,2020-09-18,saylor,Bitcoin is a swarm of cyber hornets serving th...,195494.0
3922,2021-05-19,saylor,"Entities I control have now acquired 111,000 B...",99384.0
4128,2021-06-04,jack,Square is considering making a hardware wallet...,82360.5


In [35]:
total = len(df_en)

In [36]:
total

10217

In [51]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import re

drive_path = '/content/drive/MyDrive/FYP_DATA/'

df_en = pd.read_csv(drive_path + 'Bitcoin_Tweets_Scored.csv')
df_en['Date'] = pd.to_datetime(df_en['Date'])
print(f"Loaded: {df_en.shape}")
print(f"Before scoring - positive mean: {df_en['sentiment_positive'].mean():.4f}")

print("Loading FinBERT...")
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model     = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
model.eval()
print("FinBERT ready")


def score_batch(texts, batch_size=32):
    all_pos, all_neg, all_neu = [], [], []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    for i in range(total_batches):
        if i % 50 == 0:
            print(f"  Progress: {i*batch_size}/{len(texts)}", end='\r')
        start = i * batch_size
        end   = min(start + batch_size, len(texts))
        batch = texts[start:end]
        try:
            inputs = tokenizer(batch, truncation=True, padding=True,
                               max_length=512, return_tensors='pt')
            with torch.no_grad():
                outputs = model(**inputs)
            probs     = F.softmax(outputs.logits, dim=-1).numpy()
            label_map = {v: k for k, v in model.config.id2label.items()}
            pos_idx   = label_map.get('positive', 0)
            neg_idx   = label_map.get('negative', 1)
            neu_idx   = label_map.get('neutral',  2)
            all_pos.extend(probs[:, pos_idx].tolist())
            all_neg.extend(probs[:, neg_idx].tolist())
            all_neu.extend(probs[:, neu_idx].tolist())
        except Exception as e:
            print(f"\n  Batch {i} failed: {e}")
            batch_len = end - start
            all_pos.extend([1/3] * batch_len)
            all_neg.extend([1/3] * batch_len)
            all_neu.extend([1/3] * batch_len)
    print(f"\n  Done: {len(texts)}/{len(texts)}")
    return all_pos, all_neg, all_neu

texts = df_en['clean_text'].tolist()
print(f"Scoring {len(texts):,} tweets...")
pos_scores, neg_scores, neu_scores = score_batch(texts, batch_size=32)


df_en['sentiment_positive'] = pos_scores
df_en['sentiment_negative'] = neg_scores
df_en['sentiment_neutral']  = neu_scores

print(f"After scoring - positive mean: {df_en['sentiment_positive'].mean():.4f}")
print(f"After scoring - negative mean: {df_en['sentiment_negative'].mean():.4f}")
print(f"After scoring - neutral mean:  {df_en['sentiment_neutral'].mean():.4f}")

assert df_en['sentiment_positive'].mean() > 0.01, "ERROR: Scores still zero - do not save"
print("Scores verified as real")

df_en.to_csv(drive_path + 'Bitcoin_Tweets_Scored.csv', index=False)
print("Saved to Drive")

verify = pd.read_csv(drive_path + 'Bitcoin_Tweets_Scored.csv')
print(f"File verification:")
print(f"  positive mean: {verify['sentiment_positive'].mean():.4f}")
print(f"  negative mean: {verify['sentiment_negative'].mean():.4f}")
print(f"  neutral mean:  {verify['sentiment_neutral'].mean():.4f}")
print()
print("DONE - File saved with real scores")

Loaded: (10217, 19)
Before scoring - positive mean: 0.0000
Loading FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT ready
Scoring 10,217 tweets...

  Done: 10217/10217
After scoring - positive mean: 0.1686
After scoring - negative mean: 0.0700
After scoring - neutral mean:  0.7614
Scores verified as real
Saved to Drive
File verification:
  positive mean: 0.1686
  negative mean: 0.0700
  neutral mean:  0.7614

DONE - File saved with real scores


In [53]:
def weighted_daily_aggregation(group):
    weights = group['engagement_weight'].values
    weights_normalized = weights / weights.sum()
    return pd.Series({
        'sentiment_positive': float(np.average(group['sentiment_positive'], weights=weights_normalized)),
        'sentiment_negative': float(np.average(group['sentiment_negative'], weights=weights_normalized)),
        'sentiment_neutral':  float(np.average(group['sentiment_neutral'],  weights=weights_normalized)),
        'tweet_count':        int(len(group)),
        'total_engagement':   float(group['engagement_weight'].sum()),
        'unique_accounts':    int(group['user_info.screen_name'].nunique())
    })

daily_sentiment = df_en.groupby('Date').apply(weighted_daily_aggregation).reset_index()
print(f"Shape: {daily_sentiment.shape}")
print(daily_sentiment.head(5).to_string())

Shape: (2175, 7)
        Date  sentiment_positive  sentiment_negative  sentiment_neutral  tweet_count  total_engagement  unique_accounts
0 2018-01-01            0.100993            0.019809           0.879199          2.0             336.0              2.0
1 2018-01-02            0.243285            0.011321           0.745394          2.0            2486.0              2.0
2 2018-01-07            0.045812            0.013694           0.940494          1.0             356.5              1.0
3 2018-01-08            0.090366            0.020888           0.888746          1.0              26.5              1.0
4 2018-01-09            0.047538            0.017906           0.934556          1.0            1106.5              1.0


In [54]:
daily_sentiment['sentiment_compound'] = (
    daily_sentiment['sentiment_positive'] -
    daily_sentiment['sentiment_negative']
)

print(f"Compound mean: {daily_sentiment['sentiment_compound'].mean():.4f}")
print(f"Compound min:  {daily_sentiment['sentiment_compound'].min():.4f}")
print(f"Compound max:  {daily_sentiment['sentiment_compound'].max():.4f}")
print()
print(daily_sentiment[['Date','sentiment_positive','sentiment_negative',
                        'sentiment_compound','tweet_count']].head(8).to_string())

Compound mean: 0.0719
Compound min:  -0.9646
Compound max:  0.9354

        Date  sentiment_positive  sentiment_negative  sentiment_compound  tweet_count
0 2018-01-01            0.100993            0.019809            0.081184          2.0
1 2018-01-02            0.243285            0.011321            0.231964          2.0
2 2018-01-07            0.045812            0.013694            0.032118          1.0
3 2018-01-08            0.090366            0.020888            0.069478          1.0
4 2018-01-09            0.047538            0.017906            0.029632          1.0
5 2018-01-13            0.044069            0.029231            0.014837          1.0
6 2018-01-14            0.039903            0.017982            0.021922          1.0
7 2018-01-17            0.307332            0.008357            0.298976          2.0


In [55]:
df_price = pd.read_csv(drive_path + 'Bitcoin_Price_PreProcessing.csv')
df_price['Date'] = pd.to_datetime(df_price['Date'])

full_range = pd.date_range(
    start=df_price['Date'].min(),
    end=df_price['Date'].max(),
    freq='D'
)

NEUTRAL  = 1.0 / 3.0
halflife = 7

df_idx = daily_sentiment.set_index('Date').reindex(full_range)

last_pos, last_neg, last_neu, last_compound = NEUTRAL, NEUTRAL, NEUTRAL, 0.0
days_since = 999
results = []

for date in full_range:
    row = df_idx.loc[date]
    if pd.notna(row['sentiment_positive']):
        last_pos      = float(row['sentiment_positive'])
        last_neg      = float(row['sentiment_negative'])
        last_neu      = float(row['sentiment_neutral'])
        last_compound = float(row['sentiment_compound'])
        days_since    = 0
        tc  = int(row['tweet_count'])
        eng = float(row['total_engagement'])
        ua  = int(row['unique_accounts'])
        if   tc >= 10 or ua >= 6: rel = 1.00
        elif tc >= 5  or ua >= 4: rel = 0.85
        elif tc >= 3  or ua >= 3: rel = 0.65
        elif tc >= 2  or ua >= 2: rel = 0.50
        else:                     rel = 0.35
        fp, fn, fnu, fc = last_pos, last_neg, last_neu, last_compound
    else:
        days_since += 1
        w  = 0.5 ** (days_since / halflife)
        fp = w * last_pos + (1 - w) * NEUTRAL
        fn = w * last_neg + (1 - w) * NEUTRAL
        fnu= w * last_neu + (1 - w) * NEUTRAL
        fc = w * last_compound
        last_pos, last_neg, last_neu, last_compound = fp, fn, fnu, fc
        tc, eng, ua = 0, 0.0, 0
        rel = max(0.05, 0.35 * (0.5 ** (days_since / halflife)))

    results.append({
        'Date':                  date,
        'sentiment_positive':    round(fp, 6),
        'sentiment_negative':    round(fn, 6),
        'sentiment_neutral':     round(fnu, 6),
        'sentiment_compound':    round(fc, 6),
        'tweet_count':           tc,
        'total_engagement':      eng,
        'unique_accounts':       ua,
        'sentiment_reliability': round(rel, 4)
    })

daily_filled = pd.DataFrame(results)

print(f"Shape: {daily_filled.shape}")
print(f"NaN:   {daily_filled.isnull().sum().sum()}")
print(f"Compound mean: {daily_filled['sentiment_compound'].mean():.4f}")
print(f"Reliability mean: {daily_filled['sentiment_reliability'].mean():.4f}")

Shape: (2871, 9)
NaN:   0
Compound mean: 0.0675
Reliability mean: 0.5495


In [56]:
df_merged = pd.merge(df_price, daily_filled, on='Date', how='inner')

print(f"Price shape:   {df_price.shape}")
print(f"Sentiment shape: {daily_filled.shape}")
print(f"Merged shape:  {df_merged.shape}")
print(f"NaN:           {df_merged.isnull().sum().sum()}")

Price shape:   (2871, 37)
Sentiment shape: (2871, 9)
Merged shape:  (2871, 45)
NaN:           0


In [57]:
print(df_merged[['Date','Close','Label',
                  'sentiment_positive','sentiment_negative',
                  'sentiment_compound','tweet_count',
                  'sentiment_reliability']].head(8).to_string())

        Date         Close  Label  sentiment_positive  sentiment_negative  sentiment_compound  tweet_count  sentiment_reliability
0 2018-02-19  11225.299805      1            0.159520            0.011906            0.147614            2                  0.500
1 2018-02-20  11403.700195      0            0.029648            0.051988           -0.022340            1                  0.350
2 2018-02-21  10690.400391      0            0.717538            0.020475            0.697063            1                  0.350
3 2018-02-22  10005.000000      1            0.037173            0.018491            0.018682            1                  0.350
4 2018-02-23  10301.099609      0            0.048776            0.012828            0.035948            2                  0.500
5 2018-02-24   9813.070312      0            0.075603            0.043044            0.032559            0                  0.317
6 2018-02-25   9664.730469      1            0.088698            0.489903           -0.401

In [58]:
df_merged.to_csv(drive_path + 'Bitcoin_Final_Dataset.csv', index=False)

verify = pd.read_csv(drive_path + 'Bitcoin_Final_Dataset.csv')
print(f"Saved shape: {verify.shape}")
print(f"NaN: {verify.isnull().sum().sum()}")
print("Phase 3 Complete")

Saved shape: (2871, 45)
NaN: 0
Phase 3 Complete
